# AI Assistant for EBA Guidelines on Loan Origination and Monitoring

## Goal
The goal of this project is to build an AI-powered question-answering assistant over a PDF document using:
- OpenAI API
- PDF text extraction
- chunking
- vector database (ChromaDB)
- semantic retrieval
- grounded answering based on retrieved context

## Document used
EBA/GL/2020/06 – Guidelines on loan origination and monitoring

# Install dependencies

In [ ]:
!pip install -q chromadb openai pypdf

# Imports and API key

In [ ]:
import base64
from pathlib import Path
from typing import Literal

from IPython.display import Audio, Image, display
from pydantic import BaseModel
from pypdf import PdfReader

from google.colab import files, userdata
from openai import OpenAI

from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [ ]:
import os

openai_api_key = userdata.get("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is missing in Colab Secrets.")

os.environ["OPENAI_API_KEY"] = openai_api_key

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Upload PDF

In [ ]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
path_to_pdf = Path(f"/content/{file_name}")

print(f"Using PDF: {path_to_pdf}")

# Text extraction

In [ ]:
pdf_reader = PdfReader(path_to_pdf)

print(f"There are {len(pdf_reader.pages)} pages in total.")

text_fragments = []

for page in pdf_reader.pages:
    page_text = page.extract_text()
    if page_text:
        text_fragments.append(page_text)

text = "\n".join(text_fragments)

print(f"There are {len(text)} characters in total.")

# Chunks

In [ ]:
def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 200) -> list[str]:
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        current_chunk = text[start:end]
        chunks.append(current_chunk)
        start = end - overlap

    return chunks

In [ ]:
text_chunks = chunk_text(text, chunk_size=1000, overlap=200)

print(f"Total chunks: {len(text_chunks)}")
print(text_chunks[0][:500])

# Create chroma db

In [ ]:
chroma_client = PersistentClient("/content/chroma")

collection_name = "eba_guidelines"

try:
    chroma_client.delete_collection(name=collection_name)
    print("Old collection deleted.")
except:
    print("No old collection found.")

In [ ]:
chroma_collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=OpenAIEmbeddingFunction(
        api_key=openai_api_key,
        model_name="text-embedding-3-large"
    )
)

print("Chroma collection is ready.")

# insert chunks

In [ ]:
ids = [str(i + 1) for i in range(len(text_chunks))]
documents = text_chunks

chroma_collection.add(
    ids=ids,
    documents=documents
)

print(f"Inserted {len(documents)} chunks into Chroma.")

# Retrieve_information

In [ ]:
def retrieve_information(prompt: str) -> str:
    semantic_search_results = chroma_collection.query(
        query_texts=[prompt],
        n_results=5,
        include=["documents"]
    )

    retrieved_chunks = semantic_search_results["documents"][0]
    context = "\n\n=====\n\n".join(retrieved_chunks)

    conversation = [
        {
            "role": "developer",
            "content": (
                "You are a knowledgeable assistant. "
                "Answer the user's question based ONLY on the provided context from the document. "
                "Be specific and concise. "
                "If the answer is not available in the provided context, say so clearly."
            )
        },
        {
            "role": "assistant",
            "content": f"Here is the retrieved context from the document:\n\n{context}"
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=conversation
    )

    return response.output_text

# test

In [ ]:
print(retrieve_information("What is the objective of the guidelines?"))

# ask_ai

In [ ]:
class AskAIResponse(BaseModel):
    prompt: str
    format: Literal["text", "image", "audio"]

# ask_ai implementation

In [ ]:
def ask_ai(question: str):
    structured_response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {
                "role": "developer",
                "content": (
                    "Extract the user's prompt for document retrieval and the preferred return format. "
                    "Return exactly two fields: prompt and format. "
                    "The format must be one of: text, image, audio. "
                    "If the user does not specify a format, default to text. "
                    "If the user asks to see, visualize, show, or generate a scene, use image. "
                    "If the user asks to hear, read aloud, or listen, use audio."
                )
            },
            {
                "role": "user",
                "content": question
            }
        ],
        text_format=AskAIResponse
    )

    parsed = structured_response.output_parsed

    print("Structured output:")
    print(parsed)

    answer = retrieve_information(parsed.prompt)

    if parsed.format == "text":
        print("\nTEXT RESPONSE:")
        print(answer)
        return answer

    elif parsed.format == "image":
        print("\nTEXT USED FOR IMAGE:")
        print(answer)

        image_response = openai_client.images.generate(
            model="gpt-image-1",
            prompt=(
                f'This is the user request: "{question}". '
                f'This is the retrieved answer from the document: "{answer}". '
                "Generate a realistic and relevant image based only on that retrieved information."
            ),
            size="1024x1024"
        )

        image_bytes = base64.b64decode(image_response.data[0].b64_json)
        display(Image(data=image_bytes))
        return image_bytes

    elif parsed.format == "audio":
        print("\nTEXT USED FOR AUDIO:")
        print(answer)

        speech_response = openai_client.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="alloy",
            input=answer
        )

        audio_path = Path("/content/last_answer.mp3")
        speech_response.write_to_file(audio_path)
        display(Audio(str(audio_path)))
        return str(audio_path)

# test cases

In [ ]:
ask_ai("What is the objective of the guidelines? Return the answer as text.")

In [ ]:
ask_ai("What does the document say about creditworthiness assessment? Give me a text answer.")

In [ ]:
ask_ai("What is said about pricing of loans? Return text.")

In [ ]:
ask_ai("What does the document say about valuation of collateral? Give me the answer as text.")

In [ ]:
ask_ai("What is described in the monitoring framework section? Return text.")

In [ ]:
ask_ai("Show me an image illustrating technology-enabled innovation for credit granting.")

In [ ]:
ask_ai("Read aloud what the document says about ESG factors in lending.")

In [ ]:
ask_ai("What does the document say about automated models for creditworthiness assessment and credit decision-making? Return the answer as text.")